
## 1. Delta Lake Fundamentals


## Q1. What is Delta Lake?


- **Delta Lake** is an `open-source storage framework` built on Parquet files. 
- It provides `ACID properties, schema enforcement, schema evolution, time travel etc...` .
- It used to build `reliable data pipelines` on top of a data lake.


## Q2. What is the difference between Data Lake, Delta Lake and Lakehouse?


| Concept | Explanation | Example |
|---|---|---|
| Data Lake | Stores raw structured and unstructured data | CSV, JSON, images, Parquet |
| Delta Lake | Adds transaction and reliability capabilities to data lake storage | Delta tables |
| Lakehouse | Combines flexibility of data lake with capability of data warehouse | Databricks Lakehouse |


## Q3. What are the components of a Delta table?


**A Delta table mainly consists of:**

- Parquet files.
- transaction log.
- Checkpoint files.
- Deletion vector files


| Component | Purpose |
|---|---|
| Parquet files | Store actual table data |
| JSON transaction log | Records committed table changes |
| Checkpoint | Stores a consolidated snapshot of transaction state |
| Deletion vectors | Identify deleted or updated rows without immediately rewriting entire Parquet files |


## Q4. What is the difference between Parquet, ORC and Delta?


| Feature | Parquet | ORC | Delta |
|---|---|---|---|
| Type | Columnar file format | Columnar file format | Table/storage framework |
| ACID properties | not performed | not performed | performed |
| Time Travel | not performed | not performed | performed |
| Schema enforcement | not Supported | not Supported | Supported |
| Schema evolution | not Supported | not Supported | Supported |


## Q5. What are ACID properties in Delta Lake?


ACID stands for:

- A – Atomicity
- C – Consistency
- I – Isolation
- D – Durability


| Property | Meaning | Explanation |
|---|---|---|
| **Atomicity** | All or Nothing | A transaction must either complete entirely or fail entirely. |
| **Consistency** | Valid Data | Every successful transaction must leave the Delta table in a valid state. |
| **Isolation** | Independent Transactions | Multiple transactions can execute concurrently without exposing partially committed changes to readers. |
| **Durability** | Permanent Changes | Once a transaction is successfully committed, its changes are persisted. |



## Q6. What is the Delta transaction log?


- The **Delta Transaction Log** is a `JSON-based transaction file` that records `every change in a Delta table`.
- It is stored in the `_delta_log directory` inside the **Delta table's storage location**.
- It helps **Delta Lake** maintain `ACID transactions, data consistency, versioning, and time travel`.


## Q7. What is a checkpoint in Delta Lake?


- A **checkpoint** is a `consolidated snapshot of the transaction log state`. 
- It is used to `improve query performance`.
- **Delta Lake** stores `checkpoints` in `Parquet format` inside the **_delta_log directory.**


## Q8. What are deletion vectors?


- **Deletion vectors** are used to **identify rows** are `deleted or updated` without immediately rewriting the `complete Parquet file`.
- They can **improve the performance** of `DELETE, UPDATE and MERGE` operations.


## Q9. What is schema enforcement?


**Schema enforcement** is used to ensures that `incoming data` follows the `target Delta table's schema`.


**Example:**


```python
orders_df.write\
    .format("delta")\
        .mode("append")\
            .saveAsTable("workspace.ecommerce_silver.orders")


## Q10. What is schema evolution?


**Schema evolution** means `change the schema of a Delta table` when **new data contains additional columns** or **compatible schema changes**.


## Q11. What is Time Travel?


- Time Travel allows us to access previous versions of a Delta table.
- Every successful transaction creates a new table version.
- Delta Lake maintains transaction history in the _delta_log directory.
- We can access previous versions using a version number or timestamp.
- DESCRIBE HISTORY displays table transaction history.
- versionAsOf reads data from a specific previous version.
- timestampAsOf reads data as of a specific timestamp.
- RESTORE TABLE restores a table to a previous version.
- Time Travel does not create a complete copy of the table for every transaction.
- It uses transaction logs and retained data files to reconstruct previous versions.
- VACUUM removes obsolete data files based on retention rules.
- Time Travel is limited by transaction-log and data-file retention.


## Q12. What is MERGE in Delta Lake?


- MERGE is used to perform INSERT, UPDATE and DELETE operations in a single transaction based on a matching condition between source and target tables.
- It is commonly used in incremental data pipelines and CDC processing.


### 1. Existing Customer Table

| customer_id | name | city |
|---|---|---|
| 101 | Arun | Chennai |

### 2. Incoming Source Data

| customer_id | name | city |
|---|---|---|
| 101 | Arun Kumar | Bangalore |
| 102 | Priya | Madurai |

### 3. After MERGE Operation

| customer_id | name | city |
|---|---|---|
| 101 | Arun Kumar | Bangalore |
| 102 | Priya | Madurai |


## Q13. What is the difference between INSERT, UPDATE, DELETE and UPSERT?


## DML Operations:


| Operation | Purpose | Example |
|---|---|---|
| INSERT | Adds new records | New customer |
| UPDATE | Modifies existing records | Customer changes city |
| DELETE | Removes matching records | Invalid customer |
| UPSERT | Updates existing and inserts new records | Daily customer feed |


**Example:**


```python
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'DELIVERED'
WHERE order_id = 101;


DELETE FROM workspace.ecommerce_silver.orders
WHERE order_id = 999;


INSERT INTO workspace.ecommerce_silver.orders
VALUES (1001, 501, 5000.00, 'PLACED');


## Q14. What happens if multiple source records match one target record in MERGE?


- If multiple source rows match the same target row for an update or delete, the MERGE can fail because the update is ambiguous.
- We should deduplicate the source data before performing MERGE.


**Example:**


```python
from pyspark.sql.window import Window 
from pyspark.sql.functions import row_number, col

window_spec = (Window.partitionBy("customer_id")    
                     .orderBy(col("updated_at")
                     .desc()))
                     
deduplicated_df = (source_df.withColumn("rn", row_number().over(window_spec))
.filter(col("rn") == 1)
.drop("rn"))


## Q15. What is SCD?


- SCD stands for Slowly Changing Dimension.
- It is used in data warehouses to manage changes in dimension data over time.


**Example:**

- A customer changes their city from Chennai to Bangalore.
- We need to decide whether to overwrite the old information or preserve historical information.



## Q16. What is SCD Type 1?


- SCD Type 1 overwrites existing records with updated values.
- It does not maintain historical records.
- Old values are permanently replaced by new values.
- It is commonly implemented using Delta Lake MERGE INTO.
- Useful for correcting customer information and maintaining the latest data


**Example**


**Before Update:**

| Customer ID | Name | City |
|---|---|---|
| 101 | Arun | Chennai |

**After Update:**

| Customer ID | Name | City |
|---|---|---|
| 101 | Arun | Bangalore |



## Q17. What is SCD Type 2?


- SCD (Slowly Changing Dimension) Type 2 is a technique used in Data Warehousing to maintain the complete historical records of data changes.
- Instead of overwriting an existing record, SCD Type 2 creates a new record whenever a particular attribute changes.


**It preserves both old and new values using additional columns such as:**

- `start_date` – When the record becomes active.
- `end_date` – When the record becomes inactive.
- `is_current` – Identifies the latest active record.


## Q18. Difference between SCD Type 1 and Type 2?


| Feature | SCD Type 1 | SCD Type 2 |
|---|---|---|
| Definition | Overwrites existing records | Maintains historical records |
| Update Method | Updates the existing row | Expires old row and inserts a new row |
| Historical Data | Not maintained | Maintained |
| Record Count | One record per customer | Multiple records per customer |
| Additional Columns | Not required | Start Date, End Date, Is Current |
| Storage | Requires less storage | Requires more storage |
| Complexity | Simple to implement | More complex to implement |
| Data Tracking | Tracks only the latest value | Tracks all historical values |
| Use Case | Correcting customer name or spelling | Tracking customer city changes |
| Delta Lake Operation | UPDATE or MERGE | MERGE + INSERT |


**Example:**


### Customer City Change


**Original Record**

| Customer ID | Name | City |
|---|---|---|
| 101 | Arun | Chennai |

**SCD Type 1 – After Update**

| Customer ID | Name | City |
|---|---|---|
| 101 | Arun | Bangalore |

**SCD Type 2 – After Update**

| Customer ID | Name | City | Is Current |
|---|---|---|---|
| 101 | Arun | Chennai | False |
| 101 | Arun | Bangalore | True |


## Q19. What is OPTIMIZE in Delta Lake?


- OPTIMIZE is used to improve Delta table performance by reorganizing data files and reducing small-file problems.
- It performs file compaction (bin-packing) by combining smaller files into larger, more efficient files.


## Q20. What is ZORDER in Delta Lake?


- ZORDER is a data-layout optimization technique that colocates related data values in the same set of files.
- It improves data skipping, reducing the number of files that Spark needs to scan for selective queries.


### Real-time example:


**In our E-Commerce project, users frequently filter orders using:**

- Customer ID.
- Order date.
We can use ZORDER to colocate data based on these columns.


**SQL implementation:**


```python
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


**Now execute:**


```python
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE customer_id = 501;



The optimized file layout can help Spark skip irrelevant files.


## Q21. What is Partitioning in Delta Lake?


- Partitioning divides a large table into physical directory structures based on one or more partition columns.
- It helps Spark eliminate irrelevant partitions when queries filter on those columns.



## Q22. Difference between OPTIMIZE, ZORDER and Partitioning?


| Feature | OPTIMIZE | ZORDER | Partitioning |
|---|---|---|---|
| Purpose | File compaction and data layout optimization | Improves data colocation | Physical data separation |
| Reduces Small Files | Yes | During OPTIMIZE | Not necessarily |
| Improves Data Skipping | Through file compaction and layout | Yes | Through partition pruning |
| Creates Directories | No | No | Yes |
| Query Performance | Improves in suitable workloads | Can improve | Can improve |
| Used With | Delta tables | Non-liquid-clustered Delta tables | Partitioned tables |
| Main Benefit | Reduces small files | Reduces unnecessary file scanning | Skips irrelevant partitions |
| Command | OPTIMIZE table_name | OPTIMIZE table_name ZORDER BY (column_name) | PARTITIONED BY (column_name) |


**Remember:**

- `OPTIMIZE` = File compaction.
- `ZORDER` = Data colocation.
- `Partitioning` = Physical directory separation.


## Q23. What is Liquid Clustering?


- Liquid Clustering is a data layout optimization technique in Delta Lake that organizes table data based on selected clustering columns to improve query performance.
- It helps Databricks skip unnecessary data files when executing queries, reducing the amount of data scanned.
- Unlike traditional partitioning and Z-Ordering, Liquid Clustering provides a more flexible way to organize data. It can adapt the physical layout as data changes.



**Create a Liquid Clustered table:**


```python
CREATE TABLE workspace.ecommerce_silver.orders_liquid
USING DELTA
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


**Enable Liquid Clustering on an existing table:**


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


**Then reorganize the existing data:**


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


OPTIMIZE FULL is supported in Databricks Runtime 16.0 and above.


**Important:**

- Changing clustering columns does not automatically recluster all previously written files. 
- Run OPTIMIZE to apply the new layout. 
- Databricks on AWS+1


## Q24. What are the advantages of Liquid Clustering?


- **No fixed partition directories:** Avoids creating large numbers of partition directories.
- **Supports high-cardinality columns:** Useful for customer IDs, product IDs and transaction IDs.
- **Flexible clustering keys**: Clustering columns can be changed using ALTER TABLE.
- **Improved data skipping:** Reduces unnecessary file scanning.
- **Less manual maintenance:** Reduces the need to manually design partition layouts.
- **Supports evolving query patterns:** Data layout can adapt when business query requirements change.


### Change clustering keys:


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (product_id, customer_id);


### Recluster existing data:


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


## Q25. Difference between Z-Ordering and Liquid Clustering?


| Feature | Z-Ordering | Liquid Clustering |
|---|---|---|
| Purpose | Improves data skipping through data colocation | Provides flexible data layout optimization |
| Data Organization | Multi-column data colocation | Multi-dimensional clustering |
| Changing Keys | Requires a new optimization strategy | Supports changing clustering keys using ALTER TABLE |
| Partitioning | Can be used with partitioned Delta tables | Incompatible with traditional partitioning |
| Small-File Compaction | Supported through OPTIMIZE | Supported through OPTIMIZE |
| Key Flexibility | Less flexible | More flexible |
| Data Layout Management | Requires more manual management | Reduces manual data-layout management |
| New Table Recommendation | Traditional approach | Recommended for many new Databricks tables |
| ZORDER BY Syntax | Supported | Not supported together |


### Z-Ordering example:


```python
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


### Liquid Clustering example:


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);

OPTIMIZE workspace.ecommerce_silver.orders;


## Q26. How do you convert an existing Z-Ordering table to Liquid Clustering?


**Suppose our existing table uses Z-Ordering:**


```python
OPTIMIZE workspace.ecommerce_silver.orders
ZORDER BY (customer_id, order_date);


We want to migrate it to Liquid Clustering.


### Step 1: Enable Liquid Clustering


```python
ALTER TABLE workspace.ecommerce_silver.orders
CLUSTER BY (customer_id, order_date);


### Step 2: Recluster existing files


```python
OPTIMIZE workspace.ecommerce_silver.orders FULL;


### Step 3: Verify table configuration


```python
DESCRIBE DETAIL workspace.ecommerce_silver.orders;


Inspect the clustering information and table metadata.


**Example:**


```python
ALTER TABLE workspace.ecommerce_silver.orders
REPLACE PARTITIONED BY WITH CLUSTER BY
(customer_id, order_date);


### Then:


```python
OPTIMIZE workspace.ecommerce_silver.orders;


- This command is version-dependent. 
- Do not claim it works on every Databricks Runtime.


## Q27. What is the Difference Between CLUSTER BY and partitionBy?


| Feature | partitionBy | CLUSTER BY |
|---|---|---|
| Technology | Traditional partitioning | Liquid Clustering |
| Directory Structure | Creates physical partition directories | Does not create fixed partition directories |
| Changing Columns | Can require expensive data rewrites | Easier to change clustering columns |
| High-Cardinality Fields | Usually unsuitable | Supported |
| Data Layout | Fixed partition boundaries | Flexible data clustering |
| Data Organization | Separates data into physical partitions | Organizes related data values together |
| Data Skipping | Uses partition pruning | Uses data skipping |
| Maintenance | Requires partition management | Reduces manual data-layout management |
| Recommendation | Suitable for workloads with predictable, low-cardinality filters | Preferred for many new Delta tables |


### Liquid Clustering:


```python
CREATE TABLE workspace.ecommerce_silver.orders_clustered
CLUSTER BY (customer_id, order_date)
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


## Q28. What is VACUUM in Delta Lake?


- VACUUM is a Delta Lake command used to remove obsolete data files that are no longer referenced by the latest version of a Delta table.
- When we perform UPDATE, DELETE, or MERGE, Delta Lake may create new data files and mark older files as obsolete. These obsolete files are retained for a period to support Time Travel and recovery.
- VACUUM permanently deletes eligible obsolete data files to free up storage space.


### SQL implementation:


```python
VACUUM workspace.ecommerce_silver.orders;


**Preview files before deleting:**


```python
VACUUM workspace.ecommerce_silver.orders DRY RUN;

Vacuum with a specified retention period:

VACUUM workspace.ecommerce_silver.orders
RETAIN 168 HOURS;


### Important interview questions:


**1. Does VACUUM delete the latest active data files?**

- No. 
- It removes obsolete files that are eligible for deletion under the retention policy.


**2. Does VACUUM delete _delta_log JSON files?**

- No. 
- Transaction-log cleanup is handled separately through Delta Lake log retention and checkpoint processing.


**3. What happens if we run VACUUM with zero hours?**

- It can delete files required by concurrent operations and older table versions.
- Databricks has a safety check against unsafe retention periods. Disabling that protection without understanding the consequences can cause data loss.


**4. What is the default retention of the transaction log?**

- **Deleted data-file retention:** 7 days.
- **Transaction-log retention:** 30 days.
- These are separate settings.


| Feature | Time Travel | VACUUM |
|---|---|---|
| Purpose | Reads previous table versions | Removes eligible obsolete data files |
| Main Function | Accesses historical table data | Deletes old, unreferenced data files |
| Usage | Supports historical analysis and recovery | Reduces storage usage |
| Data Access | Uses retained data files and transaction logs | Identifies and removes eligible obsolete files |
| Operation | Reads historical snapshots | Physically deletes eligible files |
| Historical Data | Allows access to previous versions | Can make older versions unreadable |
| Command | VERSION AS OF / TIMESTAMP AS OF | VACUUM table_name |
| Main Benefit | Historical data recovery | Storage management |


## Q29. What is CTAS in Databricks?


- **CTAS** stands for `CREATE TABLE AS SELECT`.
- It **creates and populate**s a new table using the results of a SELECT query.


### Example:


```python
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;


### Real-time example:


We have a production orders table and need a separate table containing only completed orders.


```python
CREATE TABLE workspace.ecommerce_silver.completed_orders
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE order_status = 'COMPLETED';


**Advantages:**

- Creates a new table from query results.
- Supports data transformation during table creation.
- Creates a separate physical dataset.
- Useful for analytical tables and data migration.


**Important interview point:**

- CTAS does not automatically preserve all source table properties, constraints, permissions or historical versions.
- It creates a new table based on the query result and its derived schema.



## Q30. What is DEEP CLONE?


- DEEP CLONE creates an independent copy of a Delta table, including its data files and supported table metadata.
- The cloned table can be modified independently without affecting the source table.


### Real-time example:


Our production orders table contains 10 million records.
Before testing a major transformation, we create a development copy.


```python
CREATE TABLE workspace.ecommerce_silver.orders_deep_clone
DEEP CLONE workspace.ecommerce_silver.orders;


**Advantages:**

- Copies data independently.
- Copies supported table metadata.
- Suitable for testing and migration.
- Does not depend on the source table's data files after cloning.


**Disadvantages:**

- Requires additional storage.
- More expensive than shallow cloning.


**Important:**

- A deep clone does not copy the source table's complete Delta transaction history. 
- The clone has its own independent history.


## Q31. What is SHALLOW CLONE?


- **SHALLOW CLONE** is used to creates a `new Delta table` that copies `table metadata` but references the source table's existing data files instead of copying all the data files.
- It requires less initial storage and compute than deep cloning.


### Example:


```python
CREATE TABLE workspace.ecommerce_silver.orders_shallow_clone
SHALLOW CLONE workspace.ecommerce_silver.orders;


**Advantages:**

- Faster cloning.
- Lower initial storage requirements.
- Useful for development and testing.
- Creates a separate table with independent history.



**Disadvantages:**

- References source data files.
- Data-file lifecycle and source dependency must be understood.
- Behaviour with VACUUM depends on the clone type and catalog implementation.


### Real-time example:

- A developer wants to test queries against production orders without creating a complete physical copy.
- Shallow cloning can be useful for this situation.



### Important Unity Catalog distinction:

- Unity Catalog manages shallow-clone file references differently from legacy Hive metastore tables.
- For supported Unity Catalog shallow clones, VACUUM tracks the relationships between source and clone files to protect files that are still required.
- Do not claim that VACUUM on every Unity Catalog source table automatically breaks its shallow clones.


## Q32. Difference Between CTAS, DEEP CLONE and SHALLOW CLONE?


| Feature | CTAS | DEEP CLONE | SHALLOW CLONE |
|---|---|---|---|
| Full Form | Create Table As Select | Deep Clone | Shallow Clone |
| Data Copied | Query result is materialized into a new table | Yes, data files are copied | No full data-file copy |
| Source Metadata Preserved | Not fully | Supported Delta metadata | Supported Delta metadata |
| Independent History | Yes | Yes | Yes |
| Initial Storage Cost | Depends on result size | Higher | Lower |
| Source Data-File Dependency | No | No | Yes |
| Main Use Case | Create transformed tables | Independent testing copy | Fast development copy |
| Copy Speed | Depends on query and data size | Generally slower | Generally faster |


### 1. CTAS (Create Table As Select)



**Purpose:** Creates a new table using the result of a SELECT query.


```sql
CREATE TABLE workspace.ecommerce_silver.orders_copy
AS
SELECT *
FROM workspace.ecommerce_silver.orders;
```



**Use Case:** Creating a new table with selected or transformed data.


### 2. DEEP CLONE


**Purpose:** Creates an independent Delta table by copying the source data files and supported metadata.


```sql
CREATE TABLE workspace.ecommerce_silver.orders_deep_clone
DEEP CLONE workspace.ecommerce_silver.orders;
```


**Use Case:** 

Creating an independent copy for testing, development and backup workflows.


### 3. SHALLOW CLONE


**Purpose:** Creates a new Delta table that shares the source table's existing data files initially, without copying all the data files.


```sql
CREATE TABLE workspace.ecommerce_silver.orders_shallow_clone
SHALLOW CLONE workspace.ecommerce_silver.orders;
```


**Use Case:** 

Quickly creating a development or testing copy while reducing initial storage requirements.


### Very Important Interview Points:


- CTAS creates a new table from a SELECT query.
- DEEP CLONE copies data files and supported metadata.
- SHALLOW CLONE initially shares source data files.
- Both DEEP CLONE and SHALLOW CLONE have independent table histories.
- SHALLOW CLONE depends on the availability of source data files that it references.
- DEEP CLONE is more independent because it copies the data files.


### Important: Clone History Example


Suppose the source Delta table is currently at version 10.


**When we create a clone:**

- The clone starts with its own transaction history.
- The clone does not inherit the source table's complete transaction history.
- The clone's initial version is not automatically version 10.
- Future changes to the source and clone are tracked independently.


### Interview Answer:


## Q33. Hands-on: Create Tables Using CTAS, DEEP CLONE and SHALLOW CLONE and Compare Storage and Metadata.


- CTAS creates a new table from the result of a SELECT query. 
- DEEP CLONE copies the source Delta table's data files and supported metadata into an independent table. 
- SHALLOW CLONE creates a new table without copying all the data files initially and references the source data files. 
- Both clone types maintain independent transaction histories.


### Step 1. Create the Source Table:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_source
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders;
```



**Purpose:** 

Creates the source Delta table using the existing Silver orders table.


### Step 2. Create a Table Using CTAS:



```sql
CREATE TABLE workspace.ecommerce_silver.orders_ctas
USING DELTA
AS
SELECT *
FROM workspace.ecommerce_silver.orders_source;
```


**Purpose:** 

Creates a new Delta table by materializing the SELECT query result.


### Step 3. Create a Table Using DEEP CLONE:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_deep
DEEP CLONE workspace.ecommerce_silver.orders_source;
```


**Purpose:** 

Creates an independent copy of the source Delta table by copying its data files and supported metadata.


### Step 4. Create a Table Using SHALLOW CLONE:


```sql
CREATE TABLE workspace.ecommerce_silver.orders_shallow
SHALLOW CLONE workspace.ecommerce_silver.orders_source;
```


**Purpose:** 

Creates a separate Delta table that initially references the source table's existing data files without copying all of them.


### Step 5. Compare Row Counts:


```sql
SELECT
    'SOURCE' AS table_name,
    COUNT(*) AS total_rows
FROM workspace.ecommerce_silver.orders_source

UNION ALL

SELECT
    'CTAS',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_ctas

UNION ALL

SELECT
    'DEEP CLONE',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_deep

UNION ALL

SELECT
    'SHALLOW CLONE',
    COUNT(*)
FROM workspace.ecommerce_silver.orders_shallow;
```



**Expected Result:**

All four tables should have the same row count immediately after creation, assuming no concurrent source changes.


### Step 6. Compare Table Metadata:


Run the following commands individually.


**Source Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_source;
```


**CTAS Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_ctas;
```


**DEEP CLONE Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_deep;
```


**SHALLOW CLONE Table:**


```sql
DESCRIBE DETAIL workspace.ecommerce_silver.orders_shallow;
```


### Step 7. Compare Delta Table History:


**Source Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_source;
```


**CTAS Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_ctas;
```


**DEEP CLONE Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_deep;
```


**SHALLOW CLONE Table:**


```sql
DESCRIBE HISTORY workspace.ecommerce_silver.orders_shallow;
```


**Observation:**

- Each table maintains its own transaction history. 
- A clone does not inherit the complete source transaction history.


### Step 8. Final Conclusion:

- CTAS creates a new table using query results.
- DEEP CLONE creates an independent copy of the source data files and supported metadata.
- SHALLOW CLONE creates a separate table without initially copying all source data files.
- All three approaches create separate table identities and transaction histories.
- SHALLOW CLONE requires the referenced source data files to remain available until the clone no longer depends on them.


**Important:** 

- Actual storage usage depends on source size, file layout, clone behavior, incremental changes and Unity Catalog implementation. 
- Do not expect identical storage measurements in every environment.



## Q34. What is Change Data Feed (CDF) in Delta Lake?


- Change Data Feed (CDF) is a Delta Lake feature that captures row-level changes made to a Delta table, including INSERT, UPDATE and DELETE operations.
- It allows downstream pipelines to process only changed records instead of reading the entire source table repeatedly.



### Real-Time E-Commerce Example:


Suppose our E-Commerce application has a customers table.


**Initial Customers Table:**

| customer_id | name | city |
|---|---|---|
| 101 | Arun | Chennai |
| 102 | Priya | Madurai |


**Next Day Changes:**

1. Customer 101 changes city from Chennai to Bangalore.
2. Customer 102 is deleted.
3. Customer 103 is inserted.


## Q35. How do you enable Change Data Feed?


### Step 1: Enable CDF:


```sql
ALTER TABLE workspace.ecommerce_silver.customers
SET TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
);
```


**Important:**

- This enables legacy, write-time-materialized CDF for subsequent changes.
- It does not retroactively capture changes made before enabling CDF.


### Step 2: Insert a record:


```sql
INSERT INTO workspace.ecommerce_silver.customers
VALUES (103, 'Rahul', 'Coimbatore');
```


### Step 3: Update a record:


```sql
UPDATE workspace.ecommerce_silver.customers
SET city = 'Bangalore'
WHERE customer_id = 101;
```


### Step 4: Delete a record:


```sql
DELETE FROM workspace.ecommerce_silver.customers
WHERE customer_id = 102;
```


### Step 5: Read CDF changes:


```sql
cdf_df = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 1)
    .table("workspace.ecommerce_silver.customers")
)

display(cdf_df)
```


Choose a starting version that is available and at or after the version where CDF was enabled.


### CDF Metadata Columns:


CDF returns additional metadata columns that help identify when and how each record was changed.


| Column | Purpose |
|---|---|
| _change_type | Identifies the type of change |
| _commit_version | Identifies the Delta table version in which the change occurred |
| _commit_timestamp | Identifies the timestamp of the commit |



### Possible _change_type Values:


| _change_type | Meaning | Example |
|---|---|---|
| insert | A new record is inserted | New customer added |
| update_preimage | Old record before an update | Chennai |
| update_postimage | New record after an update | Bangalore |
| delete | An existing record is deleted | Customer removed |


### Example: CDF Output:


| customer_id | name | city | _change_type | _commit_version |
|---|---|---|---|---|
| 101 | Arun | Chennai | update_preimage | 5 |
| 101 | Arun | Bangalore | update_postimage | 5 |
| 102 | Priya | Madurai | delete | 6 |
| 103 | Kumar | Theni | insert | 7 |



**Important:**

- An UPDATE can produce two records:
- update_preimage: Old row before update.
- update_postimage: New row after update.


## Q36. How do you use CDF in a real-time incremental pipeline?


Suppose the customers table is our source and the customer_audit table is our downstream table.


### Step 1: Read changes:


```sql
cdf_df = (
    spark.read
    .format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 5)
    .table("workspace.ecommerce_silver.customers")
)

display(cdf_df)
```


### Step 2: Filter inserted and updated records


```sql
from pyspark.sql.functions import col

changed_records = cdf_df.filter(
    col("_change_type").isin(
        "insert",
        "update_postimage"
    )
)
```



### Step 3: Read deleted records:


```sql
deleted_records = cdf_df.filter(
    col("_change_type") == "delete"
)
```


### Step 4: Write all changes into an audit table:


```sql
(
    cdf_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        "workspace.ecommerce_silver.customer_audit"
    )
)
```


**Production considerations:**

- Track the last successfully processed source version.
- Handle retries idempotently.
- Process inserts, updates and deletes according to business requirements.
- Use MERGE when maintaining downstream current-state tables.
- Maintain a durable audit table if permanent change history is required.


**Important:**

- CDF is not a permanent audit archive by itself.
- Its data is subject to retention policies. 
- VACUUM and transaction-log cleanup can make older changes unavailable.


## Q37. What is the difference between CDF and Time Travel in Delta Lake?


| Feature | CDF (Change Data Feed) | Time Travel |
|---|---|---|
| Purpose | Captures row-level changes between table versions | Reads previous versions of a table |
| Output | Returns changed records | Returns the complete table snapshot |
| Insert detection | Yes | Not directly |
| Update detection | Yes | Not directly |
| Delete detection | Yes | Not directly |
| Main use | Incremental ETL, CDC, auditing | Historical queries, data recovery |


### Real-time Example:


**CDF:**

A customer updates their city from Chennai to Bangalore. CDF identifies the update and provides the old and new record images.

**Time Travel:**

We can read the complete customer table as it existed before the city was updated.


### Key Difference:

- **CDF:** 

    What records changed?
- **Time Travel:** 

    What did the table look like at a particular version?


## Q38. What happens when an UPDATE operation runs in Delta Lake?


**When an UPDATE operation runs:**

- Delta Lake identifies the affected records.
- It creates a new committed table version.
- Depending on the execution strategy and table features, it may rewrite affected data files or use deletion vectors.
- The transaction log records the committed changes.
- Old data files may remain available for historical queries until retention cleanup.



**Example:**


```sql
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'COMPLETED'
WHERE order_id = 'O101';
```


The update creates a new table version while maintaining Delta Lake transaction consistency.


## Q39. Can we use UPDATE and DELETE in Parquet?

- Parquet is a columnar file format. It does not provide native transactional UPDATE and DELETE operations.
- To modify Parquet data, we generally need to read the existing data, apply transformations and rewrite the affected files or dataset.
- Delta Lake provides transactional UPDATE, DELETE and MERGE operations on top of data files.


**Example:**


```sql
UPDATE workspace.ecommerce_silver.orders
SET order_status = 'CANCELLED'
WHERE order_id = 'O101';
```


**Interview point:**

Parquet is a file format, whereas Delta Lake is a storage layer that adds transaction management and table-level operations.


## Q40. What is idempotency in Delta Lake?


- Idempotency means that executing the same data pipeline multiple times produces the same final result without creating duplicate records or changing the result incorrectly.
- In Delta Lake, idempotency is important when a pipeline fails midway and needs to be restarted.


### Real-time example:

- An orders ingestion job processes 5,000 records but fails before completing.
- When the job restarts, we don't want those 5,000 records inserted again.


**We use:**

- Stable business keys.
- Deduplication.
- MERGE operations.
- Checkpoints for streaming progress.
- Retry-safe processing logic.


**Example:**


```sql
MERGE INTO workspace.ecommerce_silver.orders AS target
USING workspace.ecommerce_bronze.orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;
```


**Important:**

- MERGE alone does not guarantee idempotency.
- The source must be deduplicated, the matching key must be reliable and retry behaviour must be considered.
- For streaming applications, checkpoints and idempotent sink operations are important.


## Q41. Can multiple users access a Delta table simultaneously?


- Yes.
- Delta Lake supports concurrent reads and writes through its transaction protocol and concurrency-control mechanisms.


### How it works:

- Multiple users can read a consistent table snapshot.
- Multiple transactions can attempt writes.
- Delta Lake validates conflicting transactions.
- Conflicting writes may fail and require retrying.


## Q42. What is Data Skipping?


- **Data Skipping** is a` Delta Lake optimization technique` that avoids reading `unnecessary data files` during query execution.
- Delta Lake collects statistics such as minimum value, maximum value, and null count for columns in data files.
- When a query contains filter conditions, Spark uses these statistics to identify which files may contain matching records and skips files that definitely cannot contain them.
- This reduces disk I/O and improves query performance.


### Real-time example:


Our orders table contains 10 million records.


**We execute:**


```sql
SELECT *
FROM workspace.ecommerce_silver.orders
WHERE customer_id = 'C101';
```


- Suppose file statistics indicate that a particular file contains customer IDs only from C200 to C300.
- Spark can skip that file because it cannot contain C101.


**Advantages:**

- Reduces unnecessary file scanning.
- Improves query performance.
- Reduces I/O operations.
- Works with Delta Lake file statistics.
- Can benefit from OPTIMIZE, Z-Ordering and Liquid Clustering.


## Q43. What is the difference between OPTIMIZE and VACUUM in Delta Lake?


| Feature | OPTIMIZE | VACUUM |
|---|---|---|
| Purpose | Reorganizes data files to improve storage layout | Removes eligible obsolete data files |
| Query performance | Can improve query performance | Does not directly improve query performance |
| Storage management | Reduces small-file overhead through compaction | Frees storage by deleting obsolete files |
| Operation | File compaction and clustering | Physical file cleanup |
| Historical files | Old files remain until eligible for cleanup | Eligible obsolete files are permanently deleted |
| Time Travel | Previous versions remain available if required files are retained | Older versions may become unreadable after required files are deleted |


**Interview shortcut:**

- **OPTIMIZE** = Improve data layout.
- **VACUUM** = Clean up obsolete files.



## Q44. What is the difference between Delta Table History and Time Travel?


### Delta Table History:


Displays the operations performed on a Delta table.


**Example:**


```python
DESCRIBE HISTORY workspace.ecommerce_silver.orders;


**It provides information such as:**

- Version number.
- Timestamp.
- Operation.
- Operation parameters.
- User information, where available.
- Operation metrics.



## Time Travel:


Allows us to query a previous version of a Delta table.


**Example:**


```python
SELECT *
FROM workspace.ecommerce_silver.orders
VERSION AS OF 5;


**We can also use timestamp-based Time Travel:**


```python
SELECT *
FROM workspace.ecommerce_silver.orders
TIMESTAMP AS OF '2026-11-17 10:00:00';


| Feature | Delta Table History | Time Travel |
|---|---|---|
| Purpose | Displays table operations and changes | Reads historical table data |
| Function | Shows transaction history | Accesses previous versions of data |
| Version Information | Displays version numbers | Uses a specific version number |
| Data Access | Does not directly retrieve historical records | Retrieves historical records |
| Main Use | Auditing and tracking changes | Historical analysis and recovery |
| Command | DESCRIBE HISTORY | VERSION AS OF / TIMESTAMP AS OF |
| Output | Operation details and metadata | Historical table records |


## Q45. Can we recover deleted data after VACUUM?



If VACUUM permanently deletes the data files required for an older table version, Delta Time Travel cannot reconstruct those records from the transaction log alone.


**Recovery requires another available source, such as:**

- Backup.
- Deep clone
- Archived data.
- Replicated copy.
- External recovery mechanism.